# Pedestrian Detection - YOLOv8 (Colab)

Notebook này chạy end-to-end trên Google Colab và lưu toàn bộ kết quả vào Google Drive.

## 1) Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2) Cấu hình tham số

- Điền `ROBOFLOW_API_KEY`
- Chỉnh `OUTPUT_DIR` để giữ tất cả artifacts trong Google Drive

In [ ]:
import os
from pathlib import Path

# ======== EDIT THESE ========
ROBOFLOW_API_KEY = "YOUR_ROBOFLOW_API_KEY"
WORKSPACE = "new-workspace-5uval"
PROJECT = "human-crowd-vbdc9"
VERSION = 1

OUTPUT_DIR = "/content/drive/MyDrive/cv_pedestrian"
MODEL = "yolov8s.pt"     # quick: yolov8n.pt | better: yolov8s.pt
EPOCHS = 50
IMGSZ = 640
BATCH = 16
TRAIN_NAME = "person_yolov8_visible_colab"
# ===========================

assert ROBOFLOW_API_KEY and ROBOFLOW_API_KEY != "YOUR_ROBOFLOW_API_KEY", "Please set ROBOFLOW_API_KEY"
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
os.environ["ROBOFLOW_API_KEY"] = ROBOFLOW_API_KEY

print("OUTPUT_DIR:", OUTPUT_DIR)

## 3) Clone project + cài thư viện

In [ ]:
%cd /content
!rm -rf computer_vision
!git clone https://github.com/<YOUR_ACCOUNT>/<YOUR_REPO>.git computer_vision
%cd /content/computer_vision

!python -m pip install -U pip
!pip install -r requirements.txt

## 4) Chạy full pipeline (download -> prepare person-only -> train -> val -> failure cases)

Cell này dùng script `src/colab_train.py` để đảm bảo output nằm trong Drive.

In [ ]:
%cd /content/computer_vision

!python src/colab_train.py \
  --project-root /content/computer_vision \
  --output-dir "$OUTPUT_DIR" \
  --api-key "$ROBOFLOW_API_KEY" \
  --workspace "$WORKSPACE" \
  --project "$PROJECT" \
  --version $VERSION \
  --model "$MODEL" \
  --epochs $EPOCHS \
  --imgsz $IMGSZ \
  --batch $BATCH \
  --train-name "$TRAIN_NAME"

## 5) Kiểm tra output trong Drive

In [ ]:
from pathlib import Path

out = Path(OUTPUT_DIR)
print("Exists:", out.exists())
for p in [
    out / "roboflow_downloads",
    out / "roboflow_downloads_person_only",
    out / "runs",
    out / "colab_run_summary.json",
]:
    print(p, "->", p.exists())

## 6) Lấy số liệu cho Chương 3-4

In [ ]:
import json
from pathlib import Path

summary_md = Path(OUTPUT_DIR) / "runs" / "report_ch3_ch4_summary.md"
val_json = Path(OUTPUT_DIR) / "runs" / "val_metrics.json"
failure_json = Path(OUTPUT_DIR) / "runs" / "failure_cases" / "summary.json"

if summary_md.exists():
    print("=== report_ch3_ch4_summary.md ===")
    print(summary_md.read_text(encoding="utf-8"))
else:
    print("Missing:", summary_md)

if val_json.exists():
    print("\n=== val_metrics.json ===")
    print(json.dumps(json.loads(val_json.read_text(encoding="utf-8")), indent=2, ensure_ascii=False)[:3000])
else:
    print("Missing:", val_json)

if failure_json.exists():
    print("\n=== failure_cases/summary.json ===")
    print(json.dumps(json.loads(failure_json.read_text(encoding="utf-8")), indent=2, ensure_ascii=False))
else:
    print("Missing:", failure_json)

## 7) (Tùy chọn) Train nhanh hơn nếu thiếu thời gian

Nếu runtime yếu hoặc gần hết giờ, bạn có thể giảm:
- `MODEL = "yolov8n.pt"`
- `EPOCHS = 20~30`
- `BATCH = 8`

Sau đó chạy lại cell train.